In [ ]:
!pip install propy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.3/290.3 kB 4.0 MB/s eta 0:00:00


In [ ]:
# from propy import PyPro
from pathlib import Path
import pandas as pd
from collections import Counter
from tqdm import tqdm
import os
import propy.PyPro as PyPro

In [2]:
current_dir = Path.cwd().parent

In [ ]:
# Función para calcular los descriptores de una secuencia
def calculate_protein_descriptors(sequence: str) -> dict:
    protein = PyPro.GetProDes(sequence)
    descriptors = {
        'AAC': protein.GetAAComp(),  # Amino Acid Composition
        'MBauto': protein.GetMoreauBrotoAuto(),  # Moreau-Broto autocorrelation
        'Moranauto': protein.GetMoranAuto(),  # Moran autocorrelation
        'Gearyauto': protein.GetGearyAuto(),  # Geary autocorrelation
        'CTD': protein.GetCTD()  # Composition, Transition, Distribution
    }
    # Convertimos los valores en un solo diccionario para facilidad de uso
    combined_descriptors = {**descriptors['AAC'],
                            **descriptors['MBauto'],
                            **descriptors['Moranauto'],
                            **descriptors['Gearyauto'],
                            **descriptors['CTD']
                            }
    return combined_descriptors

In [ ]:
# Lista de aminoácidos estándar
amino_acids_standard = set("ACDEFGHIKLMNPQRSTVWY")

# Función para encontrar letras no estándar en una secuencia
def find_non_standard_amino_acids(sequence: str) -> set:
    # Usamos un set para identificar letras únicas no estándar
    non_standard_letters = set(sequence) - amino_acids_standard
    return non_standard_letters


In [ ]:
# Función para sustituir letras no estándar por la más repetida en la secuencia
def replace_non_standard_with_most_common(sequence: str, non_standard_amino_acids: set) -> str:
    # Identificar si hay letras no estándar en la secuencia
    letters_to_replace = set(sequence) & non_standard_amino_acids

    if letters_to_replace:
        # Contar la frecuencia de las letras estándar en la secuencia
        letter_counts = Counter([aa for aa in sequence if aa in amino_acids_standard])

        if letter_counts:  # Asegurarse de que haya letras estándar para elegir la más común
            # Encontrar la letra más común
            most_common_letter, _ = letter_counts.most_common(1)[0]

            # Reemplazar todas las letras no estándar por la más común
            for letter in letters_to_replace:
                sequence = sequence.replace(letter, most_common_letter)

    return sequence

In [ ]:
df_train = pd.read_csv(str(current_dir / 'datasets/Train/train.csv'))

In [ ]:
df_valid = pd.read_csv(str(current_dir / 'datasets/Validation/valid.csv'))

In [ ]:
df_test = pd.read_csv(str(current_dir / 'datasets/Test/test.csv'))

# Calculando

In [ ]:
# Aplicar la función a todas las secuencias y unir los resultados
non_standard_amino_acids = set()
df_train['secuence'].apply(lambda seq: non_standard_amino_acids.update(find_non_standard_amino_acids(seq)))
df_valid['sequence'].apply(lambda seq: non_standard_amino_acids.update(find_non_standard_amino_acids(seq)))
df_test['secuence'].apply(lambda seq: non_standard_amino_acids.update(find_non_standard_amino_acids(seq)))

non_standard_amino_acids

set()

In [ ]:
# Aplicar la función a la columna 'sequence' para reemplazar las letras no estándar
df_train['secuencia'] = df_train['secuencia'].apply(lambda seq: replace_non_standard_with_most_common(seq, non_standard_amino_acids))
df_valid['secuencia'] = df_valid['secuencia'].apply(lambda seq: replace_non_standard_with_most_common(seq, non_standard_amino_acids))
df_test['secuencia'] = df_test['secuencia'].apply(lambda seq: replace_non_standard_with_most_common(seq, non_standard_amino_acids))


In [ ]:
# Configuración para el procesamiento en paquetes
batch_size = 40000
num_batches = len(df_train) // batch_size + (1 if len(df_train) % batch_size != 0 else 0)

In [ ]:
# Crear un directorio para guardar los archivos temporales
output_dir = str(current_dir / "datasets/Train/temp/")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Inicializar la barra de progreso
tqdm.pandas(desc="Calculando descriptores")

# Procesar en paquetes y guardar cada uno
for i in range(2, num_batches):
    start_index = i * batch_size
    end_index = min((i + 1) * batch_size, len(df_train))
    df_batch = df_train.iloc[start_index:end_index]

    # Aplicar la función y expandir los resultados en un nuevo DataFrame
    df_descriptors = df_batch['sequence'].progress_apply(calculate_protein_descriptors).apply(pd.Series)

    # Concatenar con el DataFrame original
    df_combined = pd.concat([df_batch, df_descriptors], axis=1)

    # Guardar el DataFrame combinado en un CSV temporal
    df_combined.to_csv(f"{output_dir}train_descriptores_batch_{i}.csv", index=False)

Calculando descriptores: 100%|██████████| 36832/36832 [47:02<00:00, 13.05it/s]


In [ ]:
# Unir todos los archivos temporales en un solo DataFrame
all_files = [f"{output_dir}train_descriptores_batch_{i}.csv" for i in range(num_batches)]
df_final = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)
os.makedirs(str(current_dir / "datasets/First Level/"), exist_ok=True)
# Guardar el DataFrame final
df_final.to_csv(str(current_dir / "datasets/First Level/train_descriptores.csv"), index=False)

In [ ]:
# Inicializar la barra de progreso
tqdm.pandas(desc="Calculando descriptores")

# Aplicar la función con la barra de progreso y expandir los resultados en un nuevo DataFrame
df_descriptors = df_valid['sequence'].progress_apply(calculate_protein_descriptors).apply(pd.Series)

# Concatenar con el DataFrame original si quieres mantener la columna 'sequence'
df_final = pd.concat([df_valid, df_descriptors], axis=1)

df_final.to_csv(str(current_dir / "datasets/First Level/valid_descriptores.csv"), index=False)

In [ ]:
# Inicializar la barra de progreso
tqdm.pandas(desc="Calculando descriptores")

# Aplicar la función con la barra de progreso y expandir los resultados en un nuevo DataFrame
df_descriptors = df_test['sequence'].progress_apply(calculate_protein_descriptors).apply(pd.Series)

# Concatenar con el DataFrame original si quieres mantener la columna 'sequence'
df_final = pd.concat([df_test, df_descriptors], axis=1)

df_final.to_csv(str(current_dir / "datasets/First Level/test_descriptores.csv"), index=False)

In [3]:
train = pd.read_csv(str(current_dir / "datasets/First Level/train_descriptores.csv"))
train.head(5)

,id_uniprot,sequence,ec_number,IsEnzime,A,R,N,D,C,E,...,_HydrophobicityD2001,_HydrophobicityD2025,_HydrophobicityD2050,_HydrophobicityD2075,_HydrophobicityD2100,_HydrophobicityD3001,_HydrophobicityD3025,_HydrophobicityD3050,_HydrophobicityD3075,_HydrophobicityD3100
0,C0HLW5,TDAVANGVHAISGVVDS,0.0.0.0,0.0,17.647,0.000,5.882,11.765,0.000,0.000,...,5.882,17.647,41.176,58.824,100.000,23.529,23.529,47.059,64.706,88.235
1,Q66C72,MAQVAKKILVTCALPYANGSIHLGHMLEHIQADIWVRFQRMRGNQV...,6.1.1.10,1.0,9.630,4.593,3.852,7.704,1.333,5.333,...,0.296,24.444,48.889,74.667,99.407,0.148,26.370,50.370,75.556,99.852
2,Q3Z4P3,MATLFIADLHLCVEEPAITAGFLRFLAGEARKADALYILGDLFEAW...,3.6.1.54,1.0,12.500,6.250,2.917,5.417,1.250,5.833,...,0.833,23.333,48.750,75.833,99.583,0.417,22.917,47.083,74.583,100.000
3,Q6YDN7,MLRRKPTRLELKLDDIEEFENIRKDLEARKKQKEDVEGVGTSDGEG...,0.0.0.0,0.0,3.529,9.412,5.882,8.235,0.000,11.765,...,7.059,47.059,56.471,77.647,95.294,1.176,12.941,30.588,71.765,100.000
4,Q1JHB8,MEALKKIAGVTAAQYVTDGMTIGLGTGSTAYYFVEEIGRRVKQEGL...,5.3.1.6,1.0,7.930,2.643,1.322,8.370,0.441,5.727,...,1.322,19.824,43.612,70.925,100.000,0.441,29.075,54.185,77.533,97.357


In [3]:
valid = pd.read_csv(str(current_dir / "datasets/First Level/valid_descriptores.csv"))
valid.head(5)

,id_uniprot,sequence,ec_number,IsEnzime,A,R,N,D,C,E,...,_HydrophobicityD2001,_HydrophobicityD2025,_HydrophobicityD2050,_HydrophobicityD2075,_HydrophobicityD2100,_HydrophobicityD3001,_HydrophobicityD3025,_HydrophobicityD3050,_HydrophobicityD3075,_HydrophobicityD3100
0,A7NR45,MSKLKITYRKSAIGYSHDQKATIRSLGLRRLNSVVVHDDTPTIRGM...,0.0.0.0,0,5.333,10.667,1.333,10.667,0.000,2.667,...,2.667,20.000,44.000,73.333,93.333,1.333,30.667,46.667,64.000,97.333
1,Q5FM74,MANRNDSRNNRRNKDDIEDQLVAVNRITKVVKGGRRMRFAALVVVG...,0.0.0.0,0,8.929,8.929,5.952,5.357,0.000,5.357,...,1.190,35.119,57.738,75.595,100.000,0.595,25.000,49.405,72.024,96.429
2,Q68WM5,MIKKEIIILCGPTASGKSYLGHELAKAYGCEIINIDSMQVYREIPI...,2.5.1.75,1,5.788,4.502,5.788,4.502,1.286,6.752,...,3.537,18.328,38.585,72.669,100.000,0.322,24.116,46.945,72.347,99.357
3,Q6HJF2,MKKVDVKESAVGRETRIRKQWNEQSIFEQSIQNREGAQSFVFYEGP...,6.1.1.5,1,6.099,4.066,4.647,5.615,0.968,9.584,...,0.871,22.072,45.499,70.378,99.903,0.097,26.234,51.500,75.799,99.613
4,Q35438,MTNIRKTHPLMKIINSSFIDLPAPSNISAWWNFGSLLGICLILQIL...,0.0.0.0,0,6.332,2.111,3.958,3.166,1.055,1.319,...,0.528,22.955,45.910,69.921,97.625,0.264,25.858,50.660,78.628,100.000


In [4]:
test = pd.read_csv(str(current_dir / "datasets/First Level/test_descriptores.csv"))
test.head(5)

,id_uniprot,sequence,ec_number,IsEnzime,A,R,N,D,C,E,...,_HydrophobicityD2001,_HydrophobicityD2025,_HydrophobicityD2050,_HydrophobicityD2075,_HydrophobicityD2100,_HydrophobicityD3001,_HydrophobicityD3025,_HydrophobicityD3050,_HydrophobicityD3075,_HydrophobicityD3100
0,Q03599,MQWPLIAALFTQLYHFVIGDDCYCPLCELDKLPDNCTTIHGHIQMG...,0.0.0.0,0,2.267,2.015,9.068,4.534,3.023,6.549,...,1.008,23.426,47.859,74.559,97.733,0.252,21.159,48.615,72.292,100.000
1,P0CH32,MQIFVKTLTGKTITLEVESSDTIDNVKAKIQDKEGIPPDQQRLIFA...,0.0.0.0,0,3.927,5.236,2.618,7.853,0.000,6.545,...,1.832,24.869,49.738,75.131,99.738,0.262,23.298,49.215,75.654,100.000
2,Q41224,MMTRGATRMTRTVLGHMGPRYFSTAIFRNDAGTGVMSGAAVFMHGV...,1.10.3.11,1,9.348,5.666,3.116,5.382,1.133,5.666,...,0.850,19.830,43.909,73.654,100.000,0.283,29.462,53.824,72.238,99.150
3,Q6CUC6,MAKSPKRKNSEPEDLSSNKRSKSFEFGKVENGLEPSGSDFGSDLPT...,0.0.0.0,0,7.923,7.650,6.284,4.918,0.273,6.284,...,0.546,21.038,40.710,73.770,98.907,0.273,35.519,55.191,77.869,99.454
4,Q9TF49,MTNTRKTHPLIKIINHSFIDLPAPSNISAWWNFGSLLGLCLIIQIL...,0.0.0.0,0,5.805,2.111,3.694,2.902,1.055,1.583,...,0.528,22.691,44.591,69.393,97.625,0.264,25.858,50.660,78.892,100.000


## G18-G19

In [5]:
df_18 = pd.read_csv(str(current_dir / "datasets/GH18.csv"))
df_18["Familia"] = "GH18"
df_19 = pd.read_csv(str(current_dir / "datasets/GH19.csv"))
df_19["Familia"] = "GH19"

In [6]:
df_test = pd.concat([df_18, df_19])
df_test.head(5)

,Nombre,Secuencia,Familia
0,AAB97779.1,MRKFNKPLLALLIGSTLCSAAQAAAPGKPTIAWGNTKFAIVEVDQA...,GH18
1,AAC44673.1,EYHMGNTMTKAIYDKFKAATPYGNTVATGAVPSETVDIAVSIGGFK...,GH18
2,AAM08773.1,MVNGYLFREYIGAQFTGVRFSDVPVNPGLSFHFILAFAIDYFMATQ...,GH18
3,AAM08776.1,MTNGYLFREYIGAQFTGVRFSDVPINPNLSFNFILSFAIDYTSPAG...,GH18
4,BAB79618.1,MKQHVKTLKLSQMASILGAIFCANSYAYNCDGLPEWSQSSAYNGGA...,GH18


In [ ]:
non_standard_amino_acids = set()
df_test['Secuencia'].apply(lambda seq: non_standard_amino_acids.update(find_non_standard_amino_acids(seq)))
df_test['Secuencia'] = df_test['Secuencia'].apply(lambda seq: replace_non_standard_with_most_common(seq, non_standard_amino_acids))

In [ ]:
tqdm.pandas(desc="Calculando descriptores")

# Aplicar la función y expandir los resultados en un nuevo DataFrame
df_descriptors = df_test['Secuencia'].progress_apply(calculate_protein_descriptors).apply(pd.Series)

# Concatenar con el DataFrame original
df_combined = pd.concat([df_test, df_descriptors], axis=1)

In [ ]:
df_combined.to_csv(str(current_dir / "datasets/GH18_GH19_descriptores.csv"), index=False)